# CAISO OASIS ingestion
Run setup once, then execute any category cell below. Each cell refreshes every registered report in that category, using the shared settings. Completed days are skipped; gaps and new data are fetched with live logs.

Use a Python 3.10+ kernel with this project installed (`python -m pip install -e .`). [Pipeline guide](../docs/caiso_oasis.md) · [Report catalog and filters](../docs/caiso_oasis_reports.md).

In [ ]:
import logging
import sys
from dataclasses import asdict
from datetime import date
from pathlib import Path

import polars as pl

PROJECT_ROOT = next(
    path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (path / "pyproject.toml").is_file()
    and (path / "src/comm_research").is_dir()
)
src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from comm_research.markets.power_gas.napg.caiso import caiso_oasis_pipeline as oasis

print(f"Repository: {PROJECT_ROOT}")

## Settings
`START_DATE=None` uses each report's retention floor (up to 39 months). Dates are **Pacific trading dates**; the start is inclusive and the end exclusive. `IGNORE_END_DATE=True` uses today as the exclusive historical end. For a bounded sample, set dates and `IGNORE_END_DATE=False`.

The defaults request the full available history for all registered reports, with LMPs limited to `NODES`; a first backfill can take a long time. `DATASETS_BY_CATEGORY` optionally narrows a category. Publication delays still apply. CRR inventory requires a valid `market_name` in `REPORT_PARAMETERS`.

In [ ]:
CATEGORIES = tuple(oasis.SCRAPERS)  # Remove categories you do not want to run.
START_DATE: date | None = None
END_DATE: date | None = None
IGNORE_END_DATE = True
RETENTION_MONTHS = 39
NODES = ("TH_SP15_GEN-APND", "TH_NP15_GEN-APND")
LOG_LEVEL = logging.INFO

# Optional restrictions; an omitted category includes all its reports.
DATASETS_BY_CATEGORY: dict[str, tuple[str, ...]] = {}
# DATASETS_BY_CATEGORY = {"prices": ("dam_lmp", "rtm_lmp")}

REPORT_PARAMETERS: dict[str, dict[str, str]] = {}
# REPORT_PARAMETERS = {"crr_inventory": {"market_name": "<CAISO market name>"}}

RAW_ROOT = PROJECT_ROOT / "data/raw/caiso"
LAKE_ROOT = PROJECT_ROOT / "data/lake/power_gas/napg/caiso"
STATE_ROOT = PROJECT_ROOT / "data/state/caiso_oasis"

# Example: START_DATE = date(2026, 10, 1); END_DATE = date(2026, 10, 4)
#          IGNORE_END_DATE = False

In [ ]:
RUN_SUMMARIES = {}


def run_category(category: str):
    if category not in CATEGORIES:
        print(f"Skipped disabled category: {category}")
        return None

    # Set module defaults explicitly so rerunning notebook settings is reliable.
    oasis.DATASETS = None
    oasis.START_DATE = START_DATE
    oasis.END_DATE = END_DATE
    oasis.RETENTION_MONTHS = RETENTION_MONTHS
    oasis.LOG_LEVEL = LOG_LEVEL
    summary = oasis.run(
        categories=(category,),
        datasets=DATASETS_BY_CATEGORY.get(category),
        start_date=START_DATE,
        end_date=END_DATE,
        ignore_end_date=IGNORE_END_DATE,
        nodes=NODES,
        report_parameters=REPORT_PARAMETERS,
        raw_root=RAW_ROOT,
        lake_root=LAKE_ROOT,
        state_root=STATE_ROOT,
        log_stream=sys.stdout,
    )
    RUN_SUMMARIES[category] = asdict(summary)
    return RUN_SUMMARIES[category]

## Prices
LMPs, shadow prices, ancillary-service prices, indices, and market power mitigation.

In [ ]:
prices_summary = run_category("prices")
prices_summary

## Transmission
Usage, outages, available capacity, and wheeling transactions. Current usage is a current-day snapshot.

In [ ]:
transmission_summary = run_category("transmission")
transmission_summary

## System demand
Demand, peak, wind/solar and advisory forecasts, sufficiency forecasts, and load adjustments.

In [ ]:
system_demand_summary = run_category("system_demand")
system_demand_summary

## Energy
Schedules, system energy, flexible ramping, convergence bidding, EIM, uplift, imbalance reserve, and EDAM.

In [ ]:
energy_summary = run_category("energy")
energy_summary

## Ancillary services
Requirements, operating reserves, results, and mileage components.

In [ ]:
ancillary_services_summary = run_category("ancillary_services")
ancillary_services_summary

## Congestion revenue rights
Clearing prices, inventory, revenue adjustments, and market names. Set the inventory market-name filter above before running.

In [ ]:
congestion_revenue_rights_summary = run_category("congestion_revenue_rights")
congestion_revenue_rights_summary

## Public bids
Energy, convergence and CRR public bids, plus market bid caps. Publication delays limit eligible dates.

In [ ]:
public_bids_summary = run_category("public_bids")
public_bids_summary

## Resource adequacy
RA/minimum load, CSP offers, CPM designations, and import capability.

In [ ]:
resource_adequacy_summary = run_category("resource_adequacy")
resource_adequacy_summary

## Results
Counts cover report/date/filter requests, not rows. Check each category's `failures` list if `failed` is nonzero; no-data and incomplete dates remain eligible on rerun.

Parquet: `data/lake/power_gas/napg/caiso/{dataset}/year=YYYY/month=MM/day=DD/data.parquet`. Interval partitions use UTC dates. Each category run cleans successfully persisted staging files and retains failed inputs for recovery.

In [ ]:
counts = ("downloaded", "resumed", "skipped", "adopted", "no_data", "incomplete", "failed")
if RUN_SUMMARIES:
    results = pl.DataFrame([
        {"category": category, **{key: summary[key] for key in counts}}
        for category, summary in RUN_SUMMARIES.items()
    ])
else:
    results = pl.DataFrame(schema={"category": pl.String, **{key: pl.Int64 for key in counts}})
results